# Natural Language Processing II — Lab Project  
## Multi-Class Text Classification: A Comparative Analysis

This notebook is built to run in Google Colab or Jupyter Notebook.

It includes:
- dataset loading and inspection
- exploratory data analysis (EDA)
- three preprocessing variants: raw, extreme, and optimal
- TF-IDF + Logistic Regression
- TF-IDF + Deep Neural Network
- Skip-gram Word2Vec + SimpleRNN, GRU, LSTM, BiSimpleRNN, BiGRU, BiLSTM
- validation-based hyperparameter tuning
- final test-set evaluation with accuracy, macro F1, confusion matrix, and classification report

**Before running:** upload your training and test CSV files and set the file paths in the configuration cell below.

## 1) Setup

Install the libraries commonly used in Colab, then import everything needed for the project.

In [ ]:
!pip -q install gensim nltk

In [ ]:
import os
import re
import json
import math
import random
import string
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("stopwords")
nltk.download("punkt")

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2) Configuration

Set the file paths for your dataset.  
The notebook is written to handle different column names by inferring the text and label columns automatically.

In [ ]:
TRAIN_PATH = "/content/train.csv"
TEST_PATH = "/content/test.csv"

RANDOM_STATE = 42
VALIDATION_SIZE = 0.15

TFIDF_MAX_FEATURES = 6000
TFIDF_NGRAM_RANGE = (1, 2)

WORD2VEC_VECTOR_SIZE = 100
WORD2VEC_WINDOW = 5
WORD2VEC_MIN_COUNT = 2
WORD2VEC_EPOCHS = 10

MAX_SEQUENCE_LEN = 40

RUN_ALL_VARIANTS = True
RUN_ALL_MODELS = True

## 3) Helper Functions

These cells handle:
- loading the CSV files
- inferring text and label columns
- cleaning HTML
- preprocessing
- evaluation
- plotting

In [ ]:
def load_dataset(path):
    df = pd.read_csv(path)
    return df

def infer_columns(df):
    cols = list(df.columns)

    lowered = {c.lower(): c for c in cols}
    text_keywords = ["text", "headline", "title", "content", "news", "sentence", "article"]
    label_keywords = ["label", "class", "category", "target", "topic"]

    text_col = None
    label_col = None

    for key in text_keywords:
        for c in cols:
            if key in c.lower():
                text_col = c
                break
        if text_col is not None:
            break

    for key in label_keywords:
        for c in cols:
            if key in c.lower():
                label_col = c
                break
        if label_col is not None:
            break

    if text_col is None or label_col is None:
        object_cols = [c for c in cols if df[c].dtype == "object"]
        if text_col is None and object_cols:
            avg_len = {c: df[c].astype(str).str.len().mean() for c in object_cols}
            text_col = max(avg_len, key=avg_len.get)
        if label_col is None:
            non_text_cols = [c for c in cols if c != text_col]
            if non_text_cols:
                label_col = non_text_cols[-1]

    if text_col is None or label_col is None:
        raise ValueError(f"Could not infer text/label columns from: {cols}")

    return text_col, label_col

HTML_RE = re.compile(r"<.*?>")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
PUNCT_RE = re.compile(r"[^\w\s]")
DIGIT_RE = re.compile(r"\d+")
MULTISPACE_RE = re.compile(r"\s+")

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def strip_html(text):
    text = str(text)
    text = re.sub(HTML_RE, " ", text)
    text = re.sub(URL_RE, " ", text)
    return text

def preprocess_text(text, mode="none"):
    text = str(text)

    if mode == "none":
        return text

    text = strip_html(text)
    text = text.replace("&amp;", " ").replace("#39;", "'")
    text = text.lower()

    if mode == "optimal":
        text = re.sub(PUNCT_RE, " ", text)
        text = re.sub(DIGIT_RE, " ", text)
        tokens = [w for w in text.split() if w not in stop_words and len(w) > 2]
        text = " ".join(tokens)

    elif mode == "extreme":
        text = re.sub(PUNCT_RE, " ", text)
        text = re.sub(DIGIT_RE, " ", text)
        tokens = [stemmer.stem(w) for w in text.split() if w not in stop_words and len(w) > 2]
        text = " ".join(tokens)

    else:
        text = re.sub(MULTISPACE_RE, " ", text).strip()

    text = re.sub(MULTISPACE_RE, " ", text).strip()
    return text

def basic_tokenize(text):
    text = strip_html(text).lower()
    text = re.sub(PUNCT_RE, " ", text)
    text = re.sub(DIGIT_RE, " ", text)
    tokens = [w for w in text.split() if len(w) > 1]
    return tokens

def evaluate_predictions(y_true, y_pred, labels=None, title=None):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1:  {f1:.4f}")
    print("\nClassification report:\n")
    print(classification_report(y_true, y_pred, digits=4))

    if title is not None:
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        disp.plot(cmap="Blues", values_format="d")
        plt.title(title)
        plt.xticks(rotation=45, ha="right")
        plt.show()

    return {"accuracy": acc, "macro_f1": f1, "confusion_matrix": cm}

def plot_class_distribution(df, label_col):
    counts = df[label_col].value_counts()
    plt.figure(figsize=(10, 5))
    plt.bar(counts.index.astype(str), counts.values)
    plt.title("Class Distribution")
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.show()

def plot_length_distribution(df, text_col, label_col):
    lengths = df[text_col].astype(str).apply(lambda x: len(strip_html(x).split()))
    plt.figure(figsize=(10, 5))
    plt.hist(lengths, bins=30)
    plt.title("Document Length Distribution")
    plt.xlabel("Number of words")
    plt.ylabel("Frequency")
    plt.show()

    by_class = df.assign(length=lengths)
    classes = by_class[label_col].unique()
    data = [by_class.loc[by_class[label_col] == c, "length"].values for c in classes]
    plt.figure(figsize=(12, 6))
    plt.boxplot(data, labels=[str(c) for c in classes], showfliers=False)
    plt.title("Text Length by Class")
    plt.xlabel("Class")
    plt.ylabel("Words per document")
    plt.xticks(rotation=45, ha="right")
    plt.show()

def top_terms_by_class(df, text_col, label_col, n_terms=10):
    tfidf = TfidfVectorizer(stop_words="english", max_features=3000, ngram_range=(1, 1))
    cleaned = df[text_col].astype(str).apply(lambda x: preprocess_text(x, mode="optimal"))
    X = tfidf.fit_transform(cleaned)
    feature_names = np.array(tfidf.get_feature_names_out())

    classes = df[label_col].unique()
    fig, axes = plt.subplots(len(classes), 1, figsize=(12, 4 * len(classes)))
    if len(classes) == 1:
        axes = [axes]

    for ax, cls in zip(axes, classes):
        idx = df[label_col] == cls
        mean_scores = np.asarray(X[idx].mean(axis=0)).ravel()
        top_idx = mean_scores.argsort()[-n_terms:][::-1]
        ax.bar(feature_names[top_idx], mean_scores[top_idx])
        ax.set_title(f"Top terms for class: {cls}")
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

def make_variant_frame(df, text_col, mode):
    out = df.copy()
    out["clean_text"] = out[text_col].astype(str).apply(lambda x: preprocess_text(x, mode=mode))
    out["raw_text"] = out[text_col].astype(str)
    out["length"] = out["clean_text"].str.split().apply(len)
    return out

def show_examples(df, label_col, n=3):
    cols = ["raw_text", "clean_text", label_col]
    display(df[cols].head(n))

def build_token_lists(texts):
    return [basic_tokenize(x) for x in texts]

def word2vec_embedding_matrix(tokenizer, w2v_model, embedding_dim):
    word_index = tokenizer.word_index
    matrix = np.zeros((len(word_index) + 1, embedding_dim), dtype=np.float32)
    for word, idx in word_index.items():
        if word in w2v_model.wv:
            matrix[idx] = w2v_model.wv[word]
        else:
            matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))
    return matrix

def pad_sequences_from_texts(tokenizer, texts, max_len):
    seqs = tokenizer.texts_to_sequences(texts)
    return tf.keras.preprocessing.sequence.pad_sequences(seqs, maxlen=max_len, padding="post", truncating="post")

def get_class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {cls: weight for cls, weight in zip(classes, weights)}

def compile_and_fit(model, X_train, y_train, X_val, y_val, epochs=10, batch_size=32, lr=1e-3, class_weight=None):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    es = callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    rlrop = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=2,
        callbacks=[es, rlrop],
        class_weight=class_weight,
    )
    return history

def predict_classes(model, X):
    probs = model.predict(X, verbose=0)
    return np.argmax(probs, axis=1)

## 4) Load the Data

In [ ]:
train_df = load_dataset(TRAIN_PATH)
test_df = load_dataset(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

text_col, label_col = infer_columns(train_df)
print("Inferred text column:", text_col)
print("Inferred label column:", label_col)

train_df.head()

## 5) Initial Inspection and EDA

In [ ]:
print("Training columns:", list(train_df.columns))
print("Test columns:", list(test_df.columns))
print("\nTraining info:")
print(train_df.info())
print("\nMissing values in train:")
print(train_df.isna().sum())
print("\nMissing values in test:")
print(test_df.isna().sum())

In [ ]:
plot_class_distribution(train_df, label_col)
plot_length_distribution(train_df, text_col, label_col)
top_terms_by_class(train_df, text_col, label_col, n_terms=10)

## 6) Preprocessing Variants

We create three versions of the text:
- **none**: no preprocessing
- **extreme**: full cleaning + stopword removal + stemming
- **optimal**: HTML removal, lowercase, punctuation cleanup, stopword removal, but no stemming

In [ ]:
train_raw = make_variant_frame(train_df, text_col, "none")
test_raw = make_variant_frame(test_df, text_col, "none")

train_extreme = make_variant_frame(train_df, text_col, "extreme")
test_extreme = make_variant_frame(test_df, text_col, "extreme")

train_optimal = make_variant_frame(train_df, text_col, "optimal")
test_optimal = make_variant_frame(test_df, text_col, "optimal")

show_examples(train_raw, label_col, n=5)
show_examples(train_extreme, label_col, n=5)
show_examples(train_optimal, label_col, n=5)

In [ ]:
variant_map = {
    "raw": (train_raw, test_raw),
    "extreme": (train_extreme, test_extreme),
    "optimal": (train_optimal, test_optimal),
}

## 7) Train/Validation Split

The training set is split again into train/validation for manual hyperparameter tuning.

In [ ]:
def split_train_val(df, label_col):
    X = df["clean_text"].astype(str).values
    y = df[label_col].astype(str).values
    return train_test_split(
        X,
        y,
        test_size=VALIDATION_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

splits = {}
for name, (tr_df, te_df) in variant_map.items():
    X_train, X_val, y_train, y_val = split_train_val(tr_df, label_col)
    splits[name] = {
        "train_df": tr_df,
        "test_df": te_df,
        "X_train": X_train,
        "X_val": X_val,
        "y_train": y_train,
        "y_val": y_val,
    }
    print(name, len(X_train), len(X_val))

## 8) TF-IDF + Logistic Regression

This is the traditional machine learning baseline.  
Hyperparameters are tuned manually using validation macro F1.

In [ ]:
def tune_logreg_tfidf(X_train, y_train, X_val, y_val, max_features=6000, ngram_range=(1, 2)):
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, lowercase=False)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec = vectorizer.transform(X_val)

    candidates = [
        {"C": 0.5, "penalty": "l2", "solver": "lbfgs"},
        {"C": 1.0, "penalty": "l2", "solver": "lbfgs"},
        {"C": 2.0, "penalty": "l2", "solver": "lbfgs"},
    ]

    label_set = np.unique(y_train)
    best = None
    results = []

    for cfg in candidates:
        model = LogisticRegression(
            C=cfg["C"],
            penalty=cfg["penalty"],
            solver=cfg["solver"],
            max_iter=3000,
            multi_class="auto",
            random_state=RANDOM_STATE,
        )
        model.fit(X_train_vec, y_train)
        pred = model.predict(X_val_vec)
        acc = accuracy_score(y_val, pred)
        f1 = f1_score(y_val, pred, average="macro")
        results.append({**cfg, "accuracy": acc, "macro_f1": f1})

        if best is None or f1 > best["macro_f1"]:
            best = {
                "model": model,
                "vectorizer": vectorizer,
                "cfg": cfg,
                "accuracy": acc,
                "macro_f1": f1,
            }

    return best, pd.DataFrame(results)

logreg_runs = {}
for variant in ["raw", "extreme", "optimal"]:
    s = splits[variant]
    best, table = tune_logreg_tfidf(
        s["X_train"], s["y_train"], s["X_val"], s["y_val"],
        max_features=TFIDF_MAX_FEATURES,
        ngram_range=TFIDF_NGRAM_RANGE,
    )
    logreg_runs[variant] = {"best": best, "table": table}
    print(f"Variant: {variant}")
    display(table.sort_values("macro_f1", ascending=False))

In [ ]:
results = []

for variant in ["raw", "extreme", "optimal"]:
    s = splits[variant]
    model_info = logreg_runs[variant]["best"]
    vec = model_info["vectorizer"]
    clf = model_info["model"]

    X_train_full = pd.concat([pd.Series(s["X_train"]), pd.Series(s["X_val"])], axis=0).astype(str).values
    y_train_full = pd.concat([pd.Series(s["y_train"]), pd.Series(s["y_val"])], axis=0).astype(str).values

    vec_final = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=TFIDF_NGRAM_RANGE, lowercase=False)
    X_train_vec = vec_final.fit_transform(X_train_full)
    X_test_vec = vec_final.transform(s["test_df"]["clean_text"].astype(str).values)

    final_clf = LogisticRegression(
        C=model_info["cfg"]["C"],
        penalty=model_info["cfg"]["penalty"],
        solver=model_info["cfg"]["solver"],
        max_iter=3000,
        multi_class="auto",
        random_state=RANDOM_STATE,
    )
    final_clf.fit(X_train_vec, y_train_full)
    test_pred = final_clf.predict(X_test_vec)

    metrics = evaluate_predictions(
        s["test_df"][label_col].astype(str).values,
        test_pred,
        labels=np.unique(train_df[label_col].astype(str).values),
        title=f"Confusion Matrix - TF-IDF + Logistic Regression ({variant})",
    )
    results.append({
        "variant": variant,
        "representation": "TF-IDF",
        "model": "Logistic Regression",
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
    })

results_df = pd.DataFrame(results)
display(results_df)

## 9) TF-IDF + Deep Neural Network

The DNN must be at least 3–4 layers deep and start with 128+ neurons.
This section tunes a few architectures manually and keeps the best validation macro F1.

In [ ]:
def build_dnn(input_dim, num_classes, layer_sizes=(256, 128, 64), dropout=0.5, lr=1e-3):
    inputs = layers.Input(shape=(input_dim,))
    x = inputs
    for i, units in enumerate(layer_sizes):
        x = layers.Dense(units, activation="relu")(x)
        x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def tune_dnn_tfidf(X_train, y_train, X_val, y_val, max_features=6000, ngram_range=(1, 2)):
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, lowercase=False)
    X_train_vec = vectorizer.fit_transform(X_train).astype(np.float32).toarray()
    X_val_vec = vectorizer.transform(X_val).astype(np.float32).toarray()

    y_train_enc, classes = pd.factorize(y_train)
    y_val_enc = np.array([np.where(classes == y)[0][0] for y in y_val])

    candidates = [
        {"layer_sizes": (256, 128, 64), "dropout": 0.5, "lr": 1e-3, "batch_size": 32},
        {"layer_sizes": (512, 256, 128), "dropout": 0.4, "lr": 1e-3, "batch_size": 32},
        {"layer_sizes": (256, 128, 64, 32), "dropout": 0.5, "lr": 5e-4, "batch_size": 64},
    ]

    best = None
    rows = []
    for cfg in candidates:
        tf.keras.backend.clear_session()
        model = build_dnn(
            input_dim=X_train_vec.shape[1],
            num_classes=len(classes),
            layer_sizes=cfg["layer_sizes"],
            dropout=cfg["dropout"],
            lr=cfg["lr"],
        )
        es = callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
        hist = model.fit(
            X_train_vec,
            y_train_enc,
            validation_data=(X_val_vec, y_val_enc),
            epochs=12,
            batch_size=cfg["batch_size"],
            verbose=0,
            callbacks=[es],
        )
        pred = np.argmax(model.predict(X_val_vec, verbose=0), axis=1)
        acc = accuracy_score(y_val_enc, pred)
        f1 = f1_score(y_val_enc, pred, average="macro")
        rows.append({**cfg, "accuracy": acc, "macro_f1": f1})

        if best is None or f1 > best["macro_f1"]:
            best = {
                "model": model,
                "vectorizer": vectorizer,
                "classes": classes,
                "cfg": cfg,
                "accuracy": acc,
                "macro_f1": f1,
            }

    return best, pd.DataFrame(rows)

dnn_runs = {}
for variant in ["raw", "extreme", "optimal"]:
    s = splits[variant]
    best, table = tune_dnn_tfidf(
        s["X_train"], s["y_train"], s["X_val"], s["y_val"],
        max_features=TFIDF_MAX_FEATURES,
        ngram_range=TFIDF_NGRAM_RANGE,
    )
    dnn_runs[variant] = {"best": best, "table": table}
    print(f"Variant: {variant}")
    display(table.sort_values("macro_f1", ascending=False))

In [ ]:
for variant in ["raw", "extreme", "optimal"]:
    s = splits[variant]
    full_text = pd.concat([pd.Series(s["X_train"]), pd.Series(s["X_val"])], axis=0).astype(str).values
    full_labels = pd.concat([pd.Series(s["y_train"]), pd.Series(s["y_val"])], axis=0).astype(str).values

    vec = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=TFIDF_NGRAM_RANGE, lowercase=False)
    X_full = vec.fit_transform(full_text).astype(np.float32).toarray()
    X_test = vec.transform(s["test_df"]["clean_text"].astype(str).values).astype(np.float32).toarray()

    y_full_enc, classes = pd.factorize(full_labels)
    test_labels = s["test_df"][label_col].astype(str).values
    test_enc = np.array([np.where(classes == y)[0][0] for y in test_labels])

    cfg = dnn_runs[variant]["best"]["cfg"]
    tf.keras.backend.clear_session()
    model = build_dnn(
        input_dim=X_full.shape[1],
        num_classes=len(classes),
        layer_sizes=cfg["layer_sizes"],
        dropout=cfg["dropout"],
        lr=cfg["lr"],
    )
    es = callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    model.fit(
        X_full,
        y_full_enc,
        validation_split=0.1,
        epochs=12,
        batch_size=cfg["batch_size"],
        verbose=2,
        callbacks=[es],
    )
    pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    metrics = evaluate_predictions(
        test_enc,
        pred,
        labels=list(range(len(classes))),
        title=f"Confusion Matrix - TF-IDF + DNN ({variant})",
    )
    results.append({
        "variant": variant,
        "representation": "TF-IDF",
        "model": "Deep Neural Network",
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
    })

results_df = pd.DataFrame(results)
display(results_df)

## 10) Skip-Gram Word2Vec Embeddings

We now train skip-gram embeddings on each preprocessing variant and feed them to RNN-based classifiers.

In [ ]:
def train_word2vec(train_texts, vector_size=100, window=5, min_count=2, epochs=10):
    token_lists = [basic_tokenize(text) for text in train_texts]
    model = Word2Vec(
        sentences=token_lists,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=max(1, os.cpu_count() - 1),
        sg=1,
        negative=10,
        seed=RANDOM_STATE,
        epochs=epochs,
    )
    return model

def build_tokenizer(texts):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(oov_token="<OOV>")
    tokenizer.fit_on_texts(texts)
    return tokenizer

def make_embedding_layer(embedding_matrix, trainable=False):
    return layers.Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=trainable,
        mask_zero=False,
    )

def build_sequence_model(kind, vocab_size, embedding_dim, embedding_matrix, num_classes, units=128, dropout=0.3, lr=1e-3):
    inputs = layers.Input(shape=(MAX_SEQUENCE_LEN,))
    x = make_embedding_layer(embedding_matrix, trainable=False)(inputs)

    if kind == "SimpleRNN":
        x = layers.SimpleRNN(units)(x)
    elif kind == "GRU":
        x = layers.GRU(units)(x)
    elif kind == "LSTM":
        x = layers.LSTM(units)(x)
    elif kind == "BiSimpleRNN":
        x = layers.Bidirectional(layers.SimpleRNN(units))(x)
    elif kind == "BiGRU":
        x = layers.Bidirectional(layers.GRU(units))(x)
    elif kind == "BiLSTM":
        x = layers.Bidirectional(layers.LSTM(units))(x)
    else:
        raise ValueError(kind)

    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
def tune_rnn_variant(train_texts, train_labels, val_texts, val_labels, variant_name):
    processed_train = np.array([preprocess_text(t, mode=variant_name) for t in train_texts])
    processed_val = np.array([preprocess_text(t, mode=variant_name) for t in val_texts])

    tokenizer = build_tokenizer(processed_train)
    train_seq = pad_sequences_from_texts(tokenizer, processed_train, MAX_SEQUENCE_LEN)
    val_seq = pad_sequences_from_texts(tokenizer, processed_val, MAX_SEQUENCE_LEN)

    w2v = train_word2vec(processed_train, vector_size=WORD2VEC_VECTOR_SIZE, window=WORD2VEC_WINDOW, min_count=WORD2VEC_MIN_COUNT, epochs=WORD2VEC_EPOCHS)
    emb_matrix = word2vec_embedding_matrix(tokenizer, w2v, WORD2VEC_VECTOR_SIZE)

    y_train_enc, classes = pd.factorize(train_labels)
    y_val_enc = np.array([np.where(classes == y)[0][0] for y in val_labels])

    model_kinds = ["SimpleRNN", "GRU", "LSTM", "BiSimpleRNN", "BiGRU", "BiLSTM"]
    candidate_cfgs = [
        {"units": 64, "dropout": 0.3, "lr": 1e-3, "batch_size": 32},
        {"units": 128, "dropout": 0.4, "lr": 1e-3, "batch_size": 32},
    ]

    best_by_model = {}
    tuning_rows = []

    for kind in model_kinds:
        best = None
        for cfg in candidate_cfgs:
            tf.keras.backend.clear_session()
            model = build_sequence_model(
                kind=kind,
                vocab_size=len(tokenizer.word_index) + 1,
                embedding_dim=WORD2VEC_VECTOR_SIZE,
                embedding_matrix=emb_matrix,
                num_classes=len(classes),
                units=cfg["units"],
                dropout=cfg["dropout"],
                lr=cfg["lr"],
            )
            es = callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
            model.fit(
                train_seq,
                y_train_enc,
                validation_data=(val_seq, y_val_enc),
                epochs=10,
                batch_size=cfg["batch_size"],
                verbose=0,
                callbacks=[es],
            )
            pred = np.argmax(model.predict(val_seq, verbose=0), axis=1)
            acc = accuracy_score(y_val_enc, pred)
            f1 = f1_score(y_val_enc, pred, average="macro")
            row = {"variant": variant_name, "model": kind, **cfg, "accuracy": acc, "macro_f1": f1}
            tuning_rows.append(row)

            if best is None or f1 > best["macro_f1"]:
                best = {
                    "model": model,
                    "cfg": cfg,
                    "accuracy": acc,
                    "macro_f1": f1,
                }

        best_by_model[kind] = {
            "best": best,
            "tokenizer": tokenizer,
            "embedding_matrix": emb_matrix,
            "classes": classes,
            "processed_train": processed_train,
            "processed_val": processed_val,
            "train_seq": train_seq,
            "val_seq": val_seq,
            "train_enc": y_train_enc,
            "val_enc": y_val_enc,
        }

    return pd.DataFrame(tuning_rows), best_by_model

rnn_tuning_results = {}
rnn_best_models = {}
for variant in ["raw", "extreme", "optimal"]:
    s = splits[variant]
    tuning_df, best_models = tune_rnn_variant(
        s["X_train"], s["y_train"], s["X_val"], s["y_val"], variant_name=variant
    )
    rnn_tuning_results[variant] = tuning_df
    rnn_best_models[variant] = best_models
    print(f"Variant: {variant}")
    display(tuning_df.sort_values(["model", "macro_f1"], ascending=[True, False]))

In [ ]:
for variant in ["raw", "extreme", "optimal"]:
    s = splits[variant]
    processed_full = np.array([preprocess_text(t, mode=variant) for t in np.concatenate([s["X_train"], s["X_val"]])])
    processed_test = np.array([preprocess_text(t, mode=variant) for t in s["test_df"]["clean_text"].astype(str).values])

    tokenizer = build_tokenizer(processed_full)
    full_seq = pad_sequences_from_texts(tokenizer, processed_full, MAX_SEQUENCE_LEN)
    test_seq = pad_sequences_from_texts(tokenizer, processed_test, MAX_SEQUENCE_LEN)

    w2v = train_word2vec(processed_full, vector_size=WORD2VEC_VECTOR_SIZE, window=WORD2VEC_WINDOW, min_count=WORD2VEC_MIN_COUNT, epochs=WORD2VEC_EPOCHS)
    emb_matrix = word2vec_embedding_matrix(tokenizer, w2v, WORD2VEC_VECTOR_SIZE)

    full_labels = np.concatenate([s["y_train"], s["y_val"]])
    y_full_enc, classes = pd.factorize(full_labels)
    test_labels = s["test_df"][label_col].astype(str).values
    test_enc = np.array([np.where(classes == y)[0][0] for y in test_labels])

    for kind in ["SimpleRNN", "GRU", "LSTM", "BiSimpleRNN", "BiGRU", "BiLSTM"]:
        cfg = rnn_best_models[variant][kind]["best"]["cfg"]
        tf.keras.backend.clear_session()
        model = build_sequence_model(
            kind=kind,
            vocab_size=len(tokenizer.word_index) + 1,
            embedding_dim=WORD2VEC_VECTOR_SIZE,
            embedding_matrix=emb_matrix,
            num_classes=len(classes),
            units=cfg["units"],
            dropout=cfg["dropout"],
            lr=cfg["lr"],
        )
        es = callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
        model.fit(
            full_seq,
            y_full_enc,
            validation_split=0.1,
            epochs=10,
            batch_size=cfg["batch_size"],
            verbose=2,
            callbacks=[es],
        )
        pred = np.argmax(model.predict(test_seq, verbose=0), axis=1)
        metrics = evaluate_predictions(
            test_enc,
            pred,
            labels=list(range(len(classes))),
            title=f"Confusion Matrix - Skip-gram + {kind} ({variant})",
        )
        results.append({
            "variant": variant,
            "representation": "Skip-gram",
            "model": kind,
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
        })

results_df = pd.DataFrame(results)
display(results_df)

## 11) Final Comparison Tables and Plots

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["macro_f1", "accuracy"], ascending=False).reset_index(drop=True)
display(results_df)

plt.figure(figsize=(12, 6))
x = np.arange(len(results_df))
plt.bar(x - 0.2, results_df["accuracy"], width=0.4, label="Accuracy")
plt.bar(x + 0.2, results_df["macro_f1"], width=0.4, label="Macro F1")
plt.xticks(x, results_df["variant"] + " | " + results_df["model"], rotation=75, ha="right")
plt.ylabel("Score")
plt.title("Model Comparison on Test Set")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_row = results_df.iloc[0]
worst_row = results_df.iloc[-1]

print("Best-performing experiment:")
print(best_row)

print("\nWorst-performing experiment:")
print(worst_row)

## 12) Save Results

The next cell saves the experiment summary as CSV so you can include it in your report.

In [ ]:
results_df.to_csv("nlp_project_results.csv", index=False)
print("Saved:", os.path.abspath("nlp_project_results.csv"))